# XL330 → surgical insertion

Apply the marked equation, then change one gain at a time.
Press **S** to end the loop and turn torque off. For simulation, focus the
PyBullet window first. Plots appear automatically. Do not use Esc.

## Connect

Set the serial port, baud and motor ID(s).

In [ ]:
import dxl_utils as dxl

lab = dxl.connect("/dev/ttyACM0", station="S", baud=1_000_000,
                  leader=2, demo=False, tissue="bone")

## $u_F=k_p(x_d-x_F)-k_d\dot x_F$

Move the surgical instrument with the leader; no hand feedback.

**Change:** `kp`: 250 → 500 → 750; `kd`: 5 → 10. `T`: 0.001 → 0.005 → 0.010.

In [ ]:
T = 0.001  # seconds; then compare 0.005 and 0.010
#TODO kp, kd, scale = 0.0, 0.0, 0.04
with lab.run("1_unilateral", leader="off", tissue="bone", dt=T) as run:
    while run.active:
        L, F = run.read()
        xdes = max(-0.02, min(0.06, scale*L.q))
        #TODO: uF 
        run.write(follower=uF, target=xdes)
        run.wait()

## $u_L=\beta\min(F_{\mathrm{contact}},0)$

Feel compression and release on withdrawal. The follower controller is provided.

**Change:** `beta`: 0.005 → 0.01 → 0.02. Then `tissue`: `"bone"` → `"liver"`, with gains unchanged.

In [ ]:
T = 0.001
beta, tissue = 0.01, "bone"
gate = dxl.ContactGate()  # provided contact/release scaffold
with lab.run("2_contact_feedback", leader="contact", tissue=tissue, dt=T) as run:
    while run.active:
        L, F = run.read()
        xdes = max(-0.02, min(0.06, 0.04*L.q))
        uF = 500*(xdes - F.x) - 10*F.xdot  # provided follower controller
        #TODO: uL = 
        run.write(leader=uL, follower=uF, target=xdes, leader_enabled=gate(L.v, F.force))
        run.wait()

## Disconnect

In [ ]:
lab.close()